In [1]:
%pip install numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# PySpark Imports
import pyspark
from pyspark.sql import SparkSession

# ML Classifier Imports
from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, PCA
from pyspark.ml.classification import OneVsRest
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.sql.functions import mean, col
from pyspark.mllib.evaluation import BinaryClassificationMetrics
import time
import os
import sys

In [3]:
# Initialize Spark session
spark = SparkSession.builder.appName("rka7 - PCA Resource") \
	.master("spark://192.168.1.2:7077") \
	.config("spark.driver.cores", "2") \
	.config("spark.driver.memory", "10g") \
	.config("spark.executor.memory", "6g") \
	.config("spark.executor.cores", "6") \
	.config("spark.executor.instances", "2") \
	.config("spark.sql.shuffle.partitions", "10") \
.getOrCreate()

24/05/06 05:25:23 WARN Utils: Your hostname, ubuntu-virtual-machine resolves to a loopback address: 127.0.1.1; using 192.168.1.107 instead (on interface ens33)
24/05/06 05:25:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/05/06 05:25:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
#Fall22Dataset
parquet_files = ["hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2021-12-12 - 2021-12-19/part-00000-d512890f-d1e9-49d5-a136-f87f0183cb4d-c000.snappy.parquet", 
                 "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2021-12-19 - 2021-12-26/part-00000-d28b031b-bff1-4e16-853a-9b7d896627e7-c000.snappy.parquet",
                "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2021-12-26 - 2022-01-02/part-00000-94d13437-ae00-4a8c-9f38-edd0196cfdee-c000.snappy.parquet",
                "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-01-02 - 2022-01-09/part-00000-745e350a-da9e-4619-bd52-8cc23bb41ad5-c000.snappy.parquet",
                "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-08-28 - 2022-09-04/part-00000-9a46dd05-4b06-4a39-a45b-5c8460b6c37b-c000.snappy.parquet",
                "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-09-04 - 2022-09-11/part-00000-ea53b0e8-d346-44e3-9a87-1f60ac35c610-c000.snappy.parquet",
                "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-09-11 - 2022-09-18/part-00000-f9afaec0-242e-41e7-906d-a42681515d75-c000.snappy.parquet",
                "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-09-18 - 2022-09-25/part-00000-9ac876be-c07d-4a18-878d-959efa26f484-c000.snappy.parquet",
                "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-09-25 - 2022-10-02/part-00000-be6d0798-554d-4c7a-9fef-d4c07aa0ce19-c000.snappy.parquet",
                "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-10-02 - 2022-10-09/part-00000-2b76f9cc-0710-45e4-9e33-98ad5808ee79-c000.snappy.parquet",
                "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-10-16 - 2022-10-23/part-00000-9aeb279c-81c6-4481-9b30-d35d4d194fea-c000.snappy.parquet",
                "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-10-23 - 2022-10-30/part-00000-23fdcfa3-9dd3-4c72-886c-e945bfcf92e1-c000.snappy.parquet",                
                "hdfs://192.168.1.2:9000/datasets-uwf-edu/UWF-ZeekDataFall22/parquet/2022-10-09 - 2022-10-16/part-00000-b2b625bc-5816-4586-b977-35f9ed4487fd-c000.snappy.parquet"]

In [5]:
# Read the parquet files into a dataframe
df = spark.read.parquet(*parquet_files)

In [6]:
#To keep in accordance w/Zeek Dataset Removing label_technique and label_binary
df = df.drop("label_technique")
df = df.drop("label_binary")

In [7]:
# List of labels to drop
labels_to_drop = ["Collection",
                  "Command and Control",
                  "Credential Access",
                  "Defense Evasion",
                  "Discovery",
                  "Execution",
                  "Initial Access",
                  "Lateral Movement",
                  "Persistence",
                  "Privilege Escalation",
                  #"Reconnaisance",
                  "Resource Development"]

# Filter out the rows with labels to drop
df = df.filter(~col("label_tactic").isin(labels_to_drop))

# Get unique labels and their counts after filtering
filtered_label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

# Show the filtered results
#filtered_label_counts.show()

In [8]:
# Drop the datetime column
df = df.withColumn("datetime", col("datetime").cast("string"))

# Define columns to index
columns_to_index = ['service', 'conn_state', 'history', 'proto', 'dest_ip_zeek', 'community_id', 'uid', 'src_ip_zeek', 'label_tactic', 'datetime']

# Impute null values with 'null' string
for column in columns_to_index:
    df = df.fillna('null', subset=[column])

# Apply StringIndexer to each column
indexers = [StringIndexer(inputCol=column, outputCol=column+"_indexed").fit(df) for column in columns_to_index]

# Chain indexers together
pipeline = Pipeline(stages=indexers)

# Fit and transform the data
df_indexed = pipeline.fit(df).transform(df)

# Drop original columns
df_indexed = df_indexed.drop(*columns_to_index)

# Show the schema of the DataFrame
#df_indexed.show()

In [9]:
# Split the data into training and test sets
train_data, test_data = df_indexed.randomSplit([0.7, 0.3], seed=42)

In [10]:
from pyspark.ml.feature import Imputer

# List of numeric column names
numeric_columns = ['resp_pkts', 'orig_ip_bytes', 'missed_bytes', 'duration', 'orig_pkts',
                   'resp_ip_bytes', 'dest_port_zeek', 'orig_bytes', 'resp_bytes',
                   'src_port_zeek', 'ts']


# Create an Imputer object
imputer = Imputer(
    inputCols=numeric_columns,
    outputCols=["{}_imputed".format(column) for column in numeric_columns]
)

# Fit the imputer to the training data
imputer_model = imputer.setStrategy("mean").fit(train_data)

# Apply the imputer to the training data
train_data_imputed = imputer_model.transform(train_data)

# Apply the imputer to the test data
test_data_imputed = imputer_model.transform(test_data)

# Show updated DataFrames
#train_data_imputed.show()
#test_data_imputed.show()

24/05/06 05:26:01 WARN DAGScheduler: Broadcasting large task binary with size 45.2 MiB


In [11]:
from pyspark.ml.feature import VectorAssembler

# List of columns to assemble
columns_to_assemble = [column for column in train_data_imputed.columns if column.endswith("_imputed")]

# Create the VectorAssembler
assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

# Transform the training DataFrame
train_data_assembled = assembler.transform(train_data_imputed)

# Transform the test DataFrame
test_data_assembled = assembler.transform(test_data_imputed)

# Select only the features and label columns for both training and test sets
train_data_assembled = train_data_assembled.select("features", "label_tactic_indexed")
test_data_assembled = test_data_assembled.select("features", "label_tactic_indexed")

# Show the schema of the assembled training DataFrame
train_data_assembled.printSchema()

# Show the schema of the assembled test DataFrame
test_data_assembled.printSchema()

root
 |-- features: vector (nullable = true)
 |-- label_tactic_indexed: double (nullable = false)

root
 |-- features: vector (nullable = true)
 |-- label_tactic_indexed: double (nullable = false)



In [12]:
from pyspark.ml.feature import StandardScaler

# Standardize data on the training set
scaler = StandardScaler(inputCol="features", outputCol="features_normalized", withMean=True, withStd=True)
scaler_model = scaler.fit(train_data_assembled)
train_data_normalized = scaler_model.transform(train_data_assembled)
train_data_normalized = train_data_normalized.select("features_normalized", "label_tactic_indexed")

24/05/06 05:26:09 WARN DAGScheduler: Broadcasting large task binary with size 45.2 MiB
24/05/06 05:26:15 WARN DAGScheduler: Broadcasting large task binary with size 45.2 MiB


In [13]:
# Apply the same transformation to the test set
test_data_normalized = scaler_model.transform(test_data_assembled)
test_data_normalized = test_data_normalized.select("features_normalized", "label_tactic_indexed")

In [14]:
# Define the PCA model
pca = PCA(k=2, inputCol="features_normalized", outputCol="pca_features")

# Fit the PCA model on the normalized training set
start_time = time.time()
pca_model = pca.fit(train_data_normalized)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/05/06 05:26:19 WARN DAGScheduler: Broadcasting large task binary with size 45.2 MiB
24/05/06 05:26:24 WARN DAGScheduler: Broadcasting large task binary with size 45.2 MiB
24/05/06 05:26:28 WARN DAGScheduler: Broadcasting large task binary with size 45.2 MiB
24/05/06 05:26:33 WARN DAGScheduler: Broadcasting large task binary with size 45.2 MiB
24/05/06 05:26:37 WARN DAGScheduler: Broadcasting large task binary with size 45.2 MiB
24/05/06 05:26:41 WARN DAGScheduler: Broadcasting large task binary with size 45.2 MiB
24/05/06 05:26:47 WARN DAGScheduler: Broadcasting large task binary with size 45.2 MiB


Execution time: 34.596004247665405 seconds


In [15]:
# Apply PCA transformation to the training and test sets
train_pca = pca_model.transform(train_data_normalized)
test_pca = pca_model.transform(test_data_normalized)

In [16]:
# Drop the normalized column and rename the pca_features column
train_pca = train_pca.drop("features_normalized").withColumnRenamed("pca_features", "features")
test_pca = test_pca.drop("features_normalized").withColumnRenamed("pca_features", "features")

# Verify the changes
#train_pca.show()
#test_pca.show()

In [17]:
svm = LinearSVC(labelCol="label_tactic_indexed", featuresCol="features", maxIter=1)

# One Vs. Rest
ovr = OneVsRest(classifier=svm, labelCol='label_tactic_indexed')

# Fit the model
start_time = time.time()

svm_model = ovr.fit(train_pca)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/05/06 05:26:52 WARN DAGScheduler: Broadcasting large task binary with size 45.2 MiB
24/05/06 05:26:57 WARN DAGScheduler: Broadcasting large task binary with size 45.2 MiB
24/05/06 05:27:04 WARN DAGScheduler: Broadcasting large task binary with size 45.3 MiB
24/05/06 05:27:07 WARN DAGScheduler: Broadcasting large task binary with size 45.3 MiB
24/05/06 05:27:10 WARN DAGScheduler: Broadcasting large task binary with size 45.3 MiB
24/05/06 05:27:13 WARN DAGScheduler: Broadcasting large task binary with size 45.3 MiB
24/05/06 05:27:16 WARN DAGScheduler: Broadcasting large task binary with size 45.3 MiB
24/05/06 05:27:18 WARN DAGScheduler: Broadcasting large task binary with size 45.3 MiB
24/05/06 05:27:21 WARN DAGScheduler: Broadcasting large task binary with size 45.3 MiB
24/05/06 05:27:24 WARN DAGScheduler: Broadcasting large task binary with size 45.3 MiB
24/05/06 05:27:26 WARN DAGScheduler: Broadcasting large task binary with size 45.3 MiB
24/05/06 05:27:29 WARN DAGScheduler: Broadc

Execution time: 95.97542834281921 seconds


In [18]:
# Make predictions
start_time = time.time()

predictions = svm_model.transform(test_pca)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.3899707794189453 seconds


In [19]:
import numpy as np

In [ ]:
# Evaluate the model
# Calculate accuracy
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="accuracy")
accuracy = evaluator_accuracy.evaluate(predictions)

# Calculate precision
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="weightedPrecision")
precision = evaluator_precision.evaluate(predictions)

# Calculate recall
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="weightedRecall")
recall = evaluator_recall.evaluate(predictions)

# Calculate F1-score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)

#Calculate FPR
evaluator_fprL = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="falsePositiveRateByLabel")
fprL_score = evaluator_fprL.evaluate(predictions)

#Calculate Weighted FPR
evaluator_fpr = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="weightedFalsePositiveRate")
fpr_score = evaluator_fpr.evaluate(predictions)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1_score)
print("FPR by Label:", fprL_score)
print("Weighted FPR:", fpr_score)

24/05/06 05:28:31 WARN DAGScheduler: Broadcasting large task binary with size 45.3 MiB
24/05/06 05:28:40 WARN DAGScheduler: Broadcasting large task binary with size 45.3 MiB
24/05/06 05:28:49 WARN DAGScheduler: Broadcasting large task binary with size 45.3 MiB
24/05/06 05:28:57 WARN DAGScheduler: Broadcasting large task binary with size 45.3 MiB


In [ ]:
from pyspark.mllib.evaluation import BinaryClassificationMetrics
from pyspark.sql import Row

# Convert DataFrame to RDD of tuples (prediction, label)
prediction_and_labels = predictions.select("prediction", "label_tactic_indexed") \
    .rdd.map(lambda row: (float(row['prediction']), float(row['label_tactic_indexed'])))


# Instantiate BinaryClassificationMetrics
metrics = BinaryClassificationMetrics(prediction_and_labels)

# Compute AUROC
auROC = metrics.areaUnderROC

# Print AUROC
print("Area under ROC = ", auROC)

In [ ]:
spark.sparkContext.stop()